# 07 · SASRec (ticket #16)

**Input** (Add Input): Kaggle Dataset `titeohihi/mixnext-splited` (output của notebook 03).

**Output** (`/kaggle/working/out/`):
- `candidates/sasrec_{val,test}_run0.parquet`
- `sasrec/sasrec_run0.pt`: checkpoint (trọng số, tham số, danh mục bài, lịch sử huấn luyện)
- `sasrec/sasrec_run0.json`: tham số, lịch sử từng epoch (loss, NDCG@10 trên 2.000 phiên val), độ chính xác trên toàn bộ val

SASRec chuyển từ `pmixer/SASRec.pytorch` (Apache-2.0, ADR 0002); nguồn và thay đổi ở `src/models/sasrec/NOTICE.md`. Tham số cố định: hidden 128, 2 khối, 2 head, dropout 0,2, lr 0,001, batch 256, maxlen 50, cross-entropy với 256 mẫu âm dùng chung. Dừng sớm khi NDCG@10 trên mẫu val không tăng 3 epoch, hoặc sau 40 phút huấn luyện.

Settings: **Accelerator: GPU** (T4 hoặc P100), bật **Internet** để clone repo. torch có sẵn trên Kaggle. Ước tính dưới 1 giờ.

Repo được clone vào `/tmp` (không phải `/kaggle/working`), nên output chỉ có thư mục `out/`.

In [ ]:
# Bootstrap: lấy code từ GitHub. Khi #16 đã merge vào main thì đổi BRANCH = "main".
BRANCH = "b"
REPO = "/tmp/MusicRecSys"
!rm -rf {REPO} && git clone -q --depth 1 -b {BRANCH} https://github.com/yashita23/MusicRecSys {REPO}
import sys; sys.path.insert(0, REPO)

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
assert torch.cuda.is_available(), "Chưa bật GPU: Settings → Accelerator → GPU" 

In [ ]:
from pathlib import Path
DATA = sorted(Path("/kaggle/input").rglob("eval_targets.parquet"))[0].parent
OUT = Path("/kaggle/working/out")
print("input:", DATA, sorted(p.name for p in DATA.iterdir()))

In [ ]:
import time
from src.models.sasrec.cli import main

t0 = time.time()
report = main(["--data", str(DATA), "--out", str(OUT)])   # huấn luyện, dừng sớm theo val, xuất candidates + checkpoint
print(f"{time.time() - t0:.0f} giây")
for p in sorted(OUT.rglob("*.*")):
    print(p.relative_to(OUT), f"{p.stat().st_size / 1e6:.1f} MB")

In [ ]:
# Lịch sử huấn luyện và NDCG@10 trên toàn bộ val, so với baseline (#9), ALS (#14), item2vec (#15)
import pandas as pd

REFERENCE_NDCG = {"popularity": 0.0040, "itemknn": 0.1297, "als": 0.0967, "item2vec": 0.0672}
print(pd.DataFrame(report["history"]).round(4).to_string(index=False))
ndcg = report["val"]["ndcg"]
print(f"\nSASRec, {len(report['history'])} epoch | NDCG@10 val {ndcg:.4f} | HitRate@10 {report['val']['hit_rate']:.4f} | MRR {report['val']['mrr']:.4f}")
for name, ref in REFERENCE_NDCG.items():
    print(f"  so với {name} {ref}: {ndcg / ref - 1:+.1%}")

## Đăng output

1. **Save Version → Save & Run All (Commit)**, giữ Accelerator GPU.
2. Tab Output của version vừa chạy → **New Dataset** tên `mixnext-sasrec`, **Visibility: Private** (dữ liệu sinh từ MPD), License bất kỳ. Dataset chứa checkpoint (tiêu chí #16) và candidates.
   - Notebook sau Add Input bộ này và tìm file bằng `Path("/kaggle/input").rglob(<tên file>)`; nạp lại mô hình bằng `SASRec.load(<đường dẫn .pt>, sessions)`.

## Kết quả trên val (chưa chạy)

Dán bảng lịch sử và dòng "SASRec, … epoch | NDCG@10 val …" vào đây sau khi chạy.